# Simulate expectations for Rubin observations of bright transients at high redshifts.

The main sections are:

### 1. Rates. 
Calculate redshift binned intrinsic rates and Rubin observable rates.

### 2. SEDs.
Simulate TDE SEDs at a range of redshifts, including Lyman break absorption

### 3. Lightcurves.
Simulate Rubin cadence lightcurves.

Notes and caveats: 
- We account for modeled transient rates as a function of redshift. These may have high uncertainty.
- We currently use luminosities distributions of local transient, which may differ from the high redshift population
- Rubin's unique cadence and colors could change detection rates in unmodeled ways
- BTS paper claims it is biased against transients specific to galactic nucclei like TDEs
- /kpc^3 must include partial coverage, ie of Northern sky, not in plane of galaxy

In [ ]:
#FIXME: remove after done developing

import importlib
import thor.utils.rates_sims
importlib.reload(thor.utils.rates_sims)

In [ ]:
# imports 
import numpy as np1
import json
from astropy import units as u
from scipy.stats import norm
from astropy.cosmology import Planck18 as cosmo

# my functions
from thor.utils.rates_sims import (
    get_volumetric_BTS_rates,
    calculate_rates_vs_redshift,
    plot_rates_vs_redshift,
    plot_rates_grid,
    get_BTS_rates_from_filtered,
    apply_z_dependent_rates,
    get_z_limit,
    calculate_rubin_detectable_rates,
    plot_observed_vs_rubin
)

# Rates

### Step 1

Gather rates information from BTS and Plasticc papers.

I try using the raw BTS survey filtered object counts to calculate my own rates. 

Ultimately, where possible I use the rates that BTS publish in their paper rather than my own calculations

In [ ]:
# https://arxiv.org/pdf/1903.11756 (plasticc paper)

# table 1
# plasticc Nevent gen = number of events generated, corresponding to true population without observational selection bias.
# these rates are for Rubin detectable events, over 10 year survey
PLASTICCrubindetectablerates = {
    'SNIa': 1.635327e7,  # z < 1.6
    'CCSN': 5.919866e7+22599840e7, # z < 2.0; combining type II and type Ib/c
    'TDE': 5.855e4, # z < 2.6
}

# table 2
# z0rate is the volumetric rate at z=0 (x10-6 years-1 Mpc-3)
# ratio_z_0_1 is the ratio of rate at z=1 / rate at z=0
PLASTICCvolumetricrates = {
    'SNIa': {'z0rate': 25, 'ratio_z_0_1': 2.8},
    'CCSN': {'z0rate': 45+19, 'ratio_z_0_1': 4.9}, 
    'TDE': {'z0rate': 1, 'ratio_z_0_1': 0.15},
}
# track down CCSN dependence on redshift

# plasticc rate citations:

# SNIa
# z<1 from Dilday 2008 (1+z)^1.5 ; z>1 from Hounsell 2018 (1+z)^-0.5 (Table 2, pg 7)

# FIXME: redshift eqn not given for CCSN in table here, search elsewhere.
# CCSN (SNII and SNIbc)
# Strogler 2015 (Table 2, pg 7)

# TDEs
# Kochanek (2016) R(z) =  10^(-5z/6) year-1 Mpc-3 (Table 2, pg 7)
# volumetric rate at z=0 from Kochanek (2018): (1.0*10^-6) * years-1 Mpc-3

In [ ]:
# https://arxiv.org/pdf/2009.01242 (BTS paper)
# https://arxiv.org/pdf/2303.06523 (also yao et al. 2023 for TDE rates)

# PART 1: my own back of envelope calculations to go from BTS saved to intrinsic rates

# objects with magpsf<19 that pass all quality cuts
# BTS paper states faintist bin (m > 18.9 mag) is incomplete. 
# otherwise saving expected number events independent of mag
BTSfiltered = {
    'SNIa': 1352,  # each rate is per 25.5 months of survey
    'CCSN': 490,
    'TDE': 8,
    # 'gap': 5,
    # 'novae': 8,
    # 'other': 2,
}

BTSefficiency = {
    'fsky': 0.35, # avg survey footprint expressed as fraction of sky
    'fext': 0.82, # average reduction in effective survey volume owing to galactic extinction
    'frec': 0.6, # average recovery efficiency of BTS survey for transients in footprint
    'fcli': 0.9 # classification efficiency. Retain subscript for apparent mag dependence.
}

# for my own back of envelope calculations
objectmeanmagnitudes = {
    'SNIa': -19.5, #estimate from BTS Fig 9  
    'CCSN': -17.0, #estimate from BTS Fig 9  
    'TDE': -19.0, #BTS paper, all but 2 < -19
}

objectmaxmagnitudes = {
    'SNIa': -20.25, 
    'CCSN': -22.5, #SLSN 
    'TDE': -22.0, 
}

# PART 2: Use BTS instrinsic rates as most reliable.

# intrinsic rates from BTS per Gpc^3 per year over whole sky 
BTSrates = {
    'SNIa': 2.35e4 * u.Unit("1 / Gpc3"), #BTS pg 16 
    'CCSN': 10.1e4* u.Unit("1 / Gpc3"), #BTS pg 16
    'TDE': 3.1e3* u.Unit("1 / Gpc3"), # pg 21 yao et al. 2023, Tde w/ Lbb>10^43 erg/s
}

### Step 1.1
 
Do my own calculations to go from filtered objects / limiting magnitude / survey length to intrinsic object rates / year / redshift bin

In [ ]:
# For reference, print the max redshift detection based on mean absolute magnitude
# Use object absolute magnitudes to get distance for detection at 19 mag and calculate volumetric detections
BTSvolumetricobservations = get_volumetric_BTS_rates(objectmeanmagnitudes, BTSfiltered)

In [ ]:
# convert filtered objects (observed and filtered by BTS) to intrinsic occurence rates, based on BTS reported efficiency
# also convert to per year rates

BTSvolumetricrates_calculated = get_BTS_rates_from_filtered(BTSvolumetricobservations, BTSefficiency)

# convert volumetric rates to binned redshift rates
BTS_calculated_rates_vs_z_naive = calculate_rates_vs_redshift(BTSvolumetricrates_calculated)

In [ ]:
# adjust for redshift dependent rates

# plasticc rate citations (Table 2, pg 7):
# SNIa: z<1 from Dilday 2008 (1+z)^1.5 ; z>1 from Hounsell 2018 (1+z)^-0.5 (Table 2, pg 7)
# CCSN: (1+z)^4.9
# TDE: 10^(-5z/6) Kochanek (2016) 

BTS_calculated_rates_vs_z = apply_z_dependent_rates(BTS_calculated_rates_vs_z_naive)

### Step 1.2

Take rates directly from BTS paper

- Our survey probes CCSNe out to approximately z < 0.05 and SNeIa out to z < 0.1, and superluminous events beyond these limits.
- BTS paper reports SNIa rate of 2.35*10^4 per Gpc^3 per year over whole sky
- CCSN BTS assumes min abs mag < -14 and infers volumetric rate of 10.1*10^4 per Gpc^3 per year

In [ ]:
BTS_rates_vs_z_naive = calculate_rates_vs_redshift(BTSrates)
BTS_rates_vs_z = apply_z_dependent_rates(BTS_rates_vs_z_naive)

### Step 1.3

Also check Plasticc rates for comparison

In [ ]:
# FIXME: repeat for plasticc rates
PLASTICC_rates_vs_z_naive = calculate_rates_vs_redshift(PLASTICCrubindetectablerates)

### Step 2: Plot intrinsic rates

In [ ]:
plot_rates_grid([
        {'rates_vs_z': BTS_rates_vs_z_naive, 'custom_title': 'Naive rates'},
        {'rates_vs_z': BTS_rates_vs_z,       'custom_title': 'Z-corrected rates'},
    ])

In [ ]:
# plot on left is my back of envelope calculation working form BTS saved objects, plot on right is BTS reported rates.
plot_rates_grid([
        {'rates_vs_z': BTS_calculated_rates_vs_z, 'custom_title': 'Estimated rates'},
        {'rates_vs_z': BTS_rates_vs_z,       'custom_title': 'BTS Rates'},
    ])

### Step 3: Detectability

In [ ]:
# calculate Rubin detectable limits for SN1a, CCSN, TDE based on mean magnitude and max magnitude

print("Rubin detection limits based on mean magnitudes:")
avg_detectable = get_z_limit(objectmeanmagnitudes, m_lim=24.5)

print("Rubin detection limits based on max magnitudes:")
max_detectable = get_z_limit(objectmaxmagnitudes, m_lim=24.5)

### Step 3.1: first cuts for relative mag within rubin limit

In [ ]:
# to get detectability as function of redshift, we need to draw from luminosity distribution.
#
# Luminosity distributions (Gaussian in peak absolute magnitude) from literature:
#
#   SNIa : M = -19.3 ± 0.3 mag
#          Betoule et al. 2014, A&A 568, A22 (JLA compilation of 740 SNIa)
#          Tight distribution — SNIa are standardizable candles.
#
#   CCSN : M = -17.5 ± 1.5 mag
#          Richardson et al. 2014, AJ 147, 118
#          Broad distribution covering IIP (faint, ~-16.8) through Ic-BL / SLSN
#          bright tails; sigma chosen to span the observed ~4 mag range at 1-sigma.
#
#   TDE  : M = -19.0 ± 1.5 mag
#          Yao et al. 2023, ApJ 955, 6  (ZTF BTS; 33 spectroscopically confirmed TDEs)
#          van Velzen et al. 2021, ApJ 908, 4  (ZTF; 17 TDEs, blackbody peak luminosities)
#          Wide distribution reflecting a factor ~100 spread in peak optical luminosity.


lum_dists = {
    'SNIa': (-19.3, 0.3),
    'CCSN': (-17.5, 1.5),
    'TDE':  (-19.0, 1.5),
}

# compute detectable rates
rubin_detectable_rates = calculate_rubin_detectable_rates(BTS_calculated_rates_vs_z, m_lim=24.5)

# print detectable fraction table
center_z = BTS_calculated_rates_vs_z['center_z']
header = f"{'Transient':<8}  " + "  ".join(f"z={z:.2f}" for z in center_z)
print(header)
print("-" * len(header))
for transient, (M_mean, M_sigma) in lum_dists.items():
    fracs = []
    for z in center_z:
        mu_z  = cosmo.distmod(z).value
        M_lim = 24.5 - mu_z
        fracs.append(norm.cdf(M_lim, loc=M_mean, scale=M_sigma))
    row = f"{transient:<8}  " + "  ".join(f"{f:.3f}  " for f in fracs)
    print(row)

print()
print("Rubin-detectable counts per bin:")
for transient in ['SNIa', 'CCSN', 'TDE']:
    counts = rubin_detectable_rates[transient]
    row = f"{transient:<8}  " + "  ".join(f"{c:.0f}" for c in counts)
    print(row)

In [ ]:
plot_rates_vs_redshift(rubin_detectable_rates, custom_title='Rubin-Detectable Rates vs Redshift', ymin=1)

### Step 3.2 Dust extinction

In [ ]:
# papers and notes

# THE RELATIONSHIP BETWEEN INFRARED, OPTICAL, AND ULTRAVIOLET EXTINCTION Cardelli et al 1989
#https://articles.adsabs.harvard.edu/pdf/1989ApJ...345..245C

# MEASURING REDDENING WITH SDSS STELLAR SPECTRA AND RECALIBRATING SFD DUST MAPS Schlafly et al 2010
# https://arxiv.org/pdf/1012.4804

# cardelli law for galactic extinction, schlafly for extinction map
# Av = 3.1 E(B-V)
# mv = mvobs - Av

#BTS paper:
# use 0.82 average reduction in effective survey volume owing to galactic extinction

# reminder: dust opaque to UV, infrared has higher optical depth. 

In [ ]:
# Dust extinction shifts the effective magnitude limit by A_lambda at the
# observed peak wavelength of each transient type:
#   m_lim_eff(z) = m_lim - A_lambda(lambda_rest * (1+z))

# Extinction law: Fitzpatrick (1999) [extinction package]
# E(B-V) sources:
#   Full-sky average  : E(B-V) ~ 0.1 mag  (BTS uses 0.82 volume reduction factor,
#                                           ~equivalent to this for a mid-latitude survey)
#   COSMOS field      : E(B-V) = 0.018    [Schlegel, Finkbeiner & Davis 1998]

# At low E(B-V) the effect is small (<0.1 mag at optical wavelengths) but grows
# toward UV rest-frame wavelengths redshifted into the optical at high-z.

EBV_FULL_SKY = 0.1    # representative mid-latitude average
EBV_COSMOS = 0.018    # low-extinction field

rubin_detectable_rates_extincted = calculate_rubin_detectable_rates(
    BTS_calculated_rates_vs_z,
    m_lim=24.5,
    ebv=EBV_FULL_SKY,
    f_sky = 0.436, # Rubin footprint
)

# show fractional reduction from extinction vs no extinction
print(f"Fractional reduction from E(B-V)={EBV_FULL_SKY} (full-sky average):")
for transient in ['SNIa', 'CCSN', 'TDE']:
    base   = rubin_detectable_rates[transient]
    extincted = rubin_detectable_rates_extincted[transient]
    with np.errstate(invalid='ignore'):
        ratio = np.where(base > 0, extincted / base, np.nan)
    row = "  ".join(f"{r:.3f}" for r in ratio)
    print(f"  {transient:<6}: {row}")

In [ ]:
# show fractional reduction from extinction vs no extinction
center_z = BTS_calculated_rates_vs_z['center_z']
header = f"{'':8}  " + "  ".join(f"z={z:.1f}" for z in center_z)
print(f"Fractional reduction from E(B-V)={EBV_FULL_SKY} (full-sky average):")
print(header)
for transient in ['SNIa', 'CCSN', 'TDE']:
    base   = rubin_detectable_rates[transient]
    extincted = rubin_detectable_rates_extincted[transient]
    with np.errstate(invalid='ignore'):
        ratio = np.where(base > 0, extincted / base, np.nan)
    row = "  ".join(f"{r:.3f}" for r in ratio)
    print(f"  {transient:<6}: {row}")

In [ ]:
# plot without and with extinction

plot_rates_grid([
        {'rates_vs_z': rubin_detectable_rates, 'custom_title': 'No Extinction'},
        {'rates_vs_z': rubin_detectable_rates_extincted, 'custom_title': f'With E(B-V)={EBV_FULL_SKY}'},
    ], ymin=1)

### step 3.3: cosmos field specific

In [ ]:
EBV_COSMOS = 0.018    # Schlegel, Finkbeiner & Davis 1998

cosmos_rates = calculate_rubin_detectable_rates(
    BTS_calculated_rates_vs_z,
    m_lim=24.5,
    ebv=EBV_COSMOS,
    f_sky=9.6/41253 #cosmos footprint
)

In [ ]:
plot_rates_vs_redshift(cosmos_rates, custom_title='COSMOS observable rates', ymin=10**(-2))

### step 3.4: estimate rates recovered

In [ ]:
#FIXME apply cuts based on recovery due to cadence

### step 3.5: summary plots

In [ ]:
# stacked histogram
from thor.utils.rates_sims import plot_rates_layered

plot_rates_layered(
    BTS_calculated_rates_vs_z,
    rubin_detectable_rates_extincted,
    cosmos_rates,
    ymin=1,
    custom_title='Transient Rates',
)

In [ ]:
# # lets recreate figure 8 in https://arxiv.org/pdf/2509.05405
# # first fetch observed tde data using the astrootter api
# # https://astro-otter.readthedocs.io/en/latest/examples/basic_usage.html#Querying-the-OTTER-Catalog

# import otter
# db = otter.Otter()
# meta = db.get_meta(save=False)
# redshifts = np.array([t.get_redshift() for t in meta if "distance" in t and len(t["distance"]) > 0])

In [ ]:
# otter provides explicit code to make the plot
# https://astro-otter.readthedocs.io/en/latest/examples/otter_paper.html

#  get the transient meta
all_transients = db.get_meta()

# now gather the redshifts
z  = []
n_z = 0
n = len(all_transients)
for t in all_transients:
    try:
        z.append(t.get_redshift())
        n_z += 1
    except KeyError:
        continue

z = np.array(z).astype(float)

print(n_z/n * 100, "% of the transients in OTTER have a redshift")

# save redshifts:
with open("data/tde_redshifts.json", "w") as f:
    json.dump(z.tolist(), f)


In [ ]:
# filter all tdes fetched
def is_optical_tde(t):                                                                                                                                          
    cls = t.get('classification')                                                                                                                               
    if not cls:                                                                                                                                                 
        return False                                                                                                                                            
    if not cls.get('unambiguous'):                                                                                                                            
        return False
    if cls.get('discovery_method') != 'optical':
        return False                                                                                                                                            
    return any(v.get('object_class') == 'TDE' for v in cls.get('value', []))
                                                                                                                                                                
optical_tdes = [t for t in all_transients if is_optical_tde(t)]                                                                                               
print(f"found {len(optical_tdes)} confident optical TDEs in OTTER")  

# collect redshifts
redshifts = []                                                                                                                                                  
for t in optical_tdes:                                                                                                                                        
    z = t.get_redshift()
    if z is None:                                                                                                                                               
        continue
    elif isinstance(z, (list, np.ndarray)):                                                                                                                     
        vals = [v for v in z if v is not None]                                                                                                                
        if vals:
            redshifts.append(float(np.median(vals)))                                                                                                            
    else:
        redshifts.append(float(z))                                                                                                                              
                                                                                                                                                            
# save
# with open("data/tde_redshifts.json", "w") as f:                                                                                                                 
#     json.dump(redshifts, f)
                                    

In [ ]:
# plot current and future tdes

with open("../../data/tde_redshifts.json") as f:
    raw = json.load(f)
tde_z_obs = np.array([z for z in raw if z is not None], dtype=float)

plot_observed_vs_rubin(tde_z_obs, BTSvolumetricrates_calculated, m_lim=24.5, ebv=EBV_FULL_SKY, f_sky=0.436)

# SEDs

### using Planck Function in astropy BlackBody

In terms of wavelength:
$$B_{\lambda}(\lambda, T) = \frac{\frac{2 h c^2}{\lambda^5}}{\exp\left(\frac{h c}{\lambda k_B T}\right) - 1}$$

Variables:
* **$B_{\lambda}$**: Spectral radiance (Intensity)
* **$h$**: Planck constant $\approx 6.626 \times 10^{-34} \, \text{J}\cdot\text{s}$
* **$c$**: Speed of light $\approx 2.998 \times 10^8 \, \text{m/s}$
* **$k_B$**: Boltzmann constant $\approx 1.381 \times 10^{-23} \, \text{J/K}$
* **$T$**: Absolute temperature in Kelvin
* **$\lambda$**: Frequency and Wavelength

> **Note on Astropy Units:** The model returns surface brightness per unit solid angle. To convert to spectral flux density ($F_\lambda$), we integrate over the solid angle $\Omega$.

In [ ]:
# TDE modelled by black body at 30,000K, plotting flux density
#
# dust extinction: E(B-V) for MW foreground dust extinction, rv for standard MW diffuse ISM
# Fitzpatrick extinction curve has "UV bump" (2175 Å) the strongest feature in the MW extinction law
# lyman alpha forest at 1216 Angstroms in rest frame, redshifted to 1216*(1+z) in observed frame
# transmission model: np.exp(-0.0036 * (lam[laf_mask] / lya) ** 3.46)

from thor.utils.lightcurve_sims import plot_blackbody_sed_with_filters
plot_blackbody_sed_with_filters(temperature_k=30000, redshifts=[0, 0.5, 1.0, 1.5, 2.0, 2.5])

# Simulated Lightcurves

In [ ]:
# simulate LSST lightcurves
# FIXME: clarify object evolution (with correct time dilation)   
# FIXME: improve rubin cadence modeling   

from thor.utils.lightcurve_sims import (simulate_lsst_lightcurve, plot_lsst_lightcurve)
                                                                                                                                                                    
# Single TDE at z=1
lc = simulate_lsst_lightcurve(                                                                                                                               
    redshift=1.0,                                                                                                                                                 
    t_peak_mjd=60000.0,                                                                                                                                             
    L_peak_erg_s=1e44,                                                                                                                                              
    T_peak_k=30000,                                                                                                                                                 
    t_rise_days=30,
    transient_class='TDE'                                                                                                                                               
)                                                                                                                                                                   

plot_lsst_lightcurve(lc, t_peak_mjd=60000.0, redshift=1.0, T_k=30000, L_peak=1e44, ymax=20, xmin=-70, xmax=50)                                                                                                                                                                                                             

## more plotting

In [ ]:
## magnitude vs redshift plot
# FIXME plot probabilistic curve of magnitude vs redshift based only on intrinsic magnitudes, set dust extinction, for range of transients

In [ ]:
# magnitude vs timescale plot but put different redshifts of tdes, snia, ccsn.
#FIXME intrinsic z=0 values look off. use abs M_r -19.1 for SN1a, -19 for CCSN, and -19 for TDE. use sn1a rise 10 days, ccsn rise 15 days, tde rise 25 days
#FIXME plot distributions not median values

# Single panel, r-band, IGM on    
from thor.utils.lightcurve_sims import plot_transient_magnitude_timescale
                                                                                                                                  
plot_transient_magnitude_timescale(                                                                                                                              
    redshifts=[0.05, 0.1, 0.2, 0.5, 0.75, 1.0, 1.5, 2.0, 2.5, 3.0],                                                                                               
    band='r',                                                                                                                                                       
    apply_igm=True,                                                                                                                                               
)     